# Audio Basics in Colab

This notebook is a Colab-friendly version of the introduction demo.
It avoids `sounddevice` and uses browser recording or file upload instead.

In [ ]:
%pip install soundfile librosa matplotlib

# Imports and Helpers

In [ ]:
import base64
import threading
from importlib import import_module

import matplotlib.pyplot as plt
import numpy as np
import soundfile as sf
from IPython.display import Audio, Javascript, display

try:
    import_module("google.colab")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

recording = None
sampling_rate = None


def ensure_mono(audio):
    audio = np.asarray(audio)
    if audio.ndim == 1:
        return audio
    return audio.mean(axis=1)


def show_audio(audio, rate):
    audio = np.asarray(audio)
    if audio.ndim > 1:
        audio = ensure_mono(audio)
    display(Audio(audio, rate=rate))


def print_audio_info(audio, rate):
    audio = np.asarray(audio)
    print("audio data shape:", audio.shape)
    print("sampling rate:", rate)
    print("dtype:", audio.dtype)
    print("last 10 samples:")
    print(audio.reshape(-1)[-10:])

# Option A: Upload an Audio File

Use this if browser microphone permission fails or you want a stable demo.

In [ ]:
def load_audio_from_upload():
    if not IN_COLAB:
        raise RuntimeError("File upload helper in this cell is intended for Colab.")

    files = import_module("google.colab.files")

    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No file was uploaded.")

    filename = next(iter(uploaded))
    audio, rate = sf.read(filename)
    audio = ensure_mono(audio)
    return audio, rate, filename

# Option B: Record Audio in the Browser

This replaces `sounddevice` in Colab.

In [ ]:
recording_done_event = threading.Event()
recorded_audio_result = {}


def record_audio_in_colab(duration=5, filename="microphone_audio.wav"):
    if not IN_COLAB:
        raise RuntimeError("Browser recording is only available in Colab.")

    output = import_module("google.colab.output")

    recording_done_event.clear()
    recorded_audio_result.clear()

    def _record_audio_callback(audio_base64):
        audio_bytes = base64.b64decode(audio_base64)
        with open(filename, "wb") as file_handle:
            file_handle.write(audio_bytes)

        audio, rate = sf.read(filename)
        audio = ensure_mono(audio)

        recorded_audio_result["audio"] = audio
        recorded_audio_result["rate"] = rate
        recorded_audio_result["filename"] = filename
        recording_done_event.set()

    output.register_callback("notebook_record_audio_callback", _record_audio_callback)

    js_code = f"""
    async function recordAudio() {{
        const stream = await navigator.mediaDevices.getUserMedia({{ audio: true }});
        const mediaRecorder = new MediaRecorder(stream);
        const audioChunks = [];

        mediaRecorder.addEventListener('dataavailable', event => {{
            audioChunks.push(event.data);
        }});

        const audioPromise = new Promise(resolve => {{
            mediaRecorder.addEventListener('stop', () => {{
                const audioBlob = new Blob(audioChunks, {{ type: 'audio/wav' }});
                const reader = new FileReader();
                reader.onloadend = () => resolve(reader.result.split(',')[1]);
                reader.readAsDataURL(audioBlob);
            }});
        }});

        mediaRecorder.start();
        await new Promise(resolve => setTimeout(resolve, {duration * 1000}));
        mediaRecorder.stop();

        const audioBase64 = await audioPromise;
        google.colab.kernel.invokeFunction(
            'notebook_record_audio_callback',
            [audioBase64],
            {{}}
        );
    }}

    recordAudio();
    """

    display(Javascript(js_code))
    print(f"Recording for {duration} seconds...")

    timeout_seconds = duration + 15
    if not recording_done_event.wait(timeout=timeout_seconds):
        raise TimeoutError(
            "Recording timed out. Check browser microphone permission and retry."
        )

    return (
        recorded_audio_result["audio"],
        recorded_audio_result["rate"],
        recorded_audio_result["filename"],
    )

# Choose One Input Method

Run one of the following cells.

In [ ]:
# Method 1: browser recording
duration = 5
recording, sampling_rate, source_name = record_audio_in_colab(duration=duration)
print("Loaded from:", source_name)

In [ ]:
# Method 2: upload a file
# recording, sampling_rate, source_name = load_audio_from_upload()
# print("Loaded from:", source_name)

# Inspect and Play Audio

In [ ]:
print_audio_info(recording, sampling_rate)
show_audio(recording, sampling_rate)

# Visualize Waveform

In [ ]:
import librosa
import librosa.display

plt.figure(figsize=(12, 3))
librosa.display.waveshow(recording, sr=sampling_rate)
plt.title("Waveform")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.show()

# Visualize Spectrogram

In [ ]:
import librosa
import librosa.display

D = librosa.amplitude_to_db(np.abs(librosa.stft(recording)), ref=np.max)

plt.figure(figsize=(10, 4))
librosa.display.specshow(D, sr=sampling_rate, x_axis="time", y_axis="log")
plt.colorbar(format="%+2.0f dB")
plt.title("Spectrogram")
plt.show()

# Save and Load Audio File

In [ ]:
output_path = "recording.wav"
sf.write(output_path, recording, sampling_rate, subtype="PCM_24")

loaded_recording, loaded_sampling_rate = sf.read(output_path)
loaded_recording = ensure_mono(loaded_recording)

print(loaded_recording.shape, loaded_sampling_rate)
show_audio(loaded_recording, loaded_sampling_rate)

# Resample

In [ ]:
import librosa

resampled_rate = 16000
resampled_recording = librosa.resample(
    recording,
    orig_sr=sampling_rate,
    target_sr=resampled_rate,
)

print("resampled shape:", resampled_recording.shape)
show_audio(resampled_recording, resampled_rate)